# Журналирование микросервисов: ELK

FastAPI отправляет структурированные JSON-логи в Logstash, тот кладёт их в Elasticsearch, Kibana показывает UI. Ноутбук идёт напрямую в ES REST API для аналитики.

## Что поднимается
- `fastapi` -> :8000 (login, users/{id}, health)
- `logstash` -> :5000 (TCP ввод)
- `elasticsearch` -> :9200
- `kibana` -> :5601

```bash
docker compose up -d
sleep 60  # ES стартует медленно
```


In [ ]:
import json, time
from datetime import datetime, timezone
from collections import Counter
import requests
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
print(f"requests={requests.__version__}, pandas={pd.__version__}")


## 1. Health-check сервиса


In [ ]:
# Generate diverse traffic so Kibana/ES have logs to show
r = requests.get("http://localhost:8000/health", timeout=5)
print("health:", r.status_code, r.json())
assert r.status_code == 200

# Login calls: 2 succeed + 1 failure
login_scenarios = [("alice", "secret123"), ("bob", "horsebattery"), ("eve", "wrong")]
for username, password in login_scenarios:
    rr = requests.post("http://localhost:8000/login",
                       params={"username": username, "password": password},
                       timeout=5)
    print(f"  login({username}):", rr.status_code, rr.json() if rr.headers.get("content-type","").startswith("application/json") else None)

# User lookups: 3 hits, 2 misses
for uid in [1, 2, 3, 99, 999]:
    rr = requests.get(f"http://localhost:8000/users/{uid}", timeout=5)
    print(f"  users/{uid}:", rr.status_code)

# Trigger one slow lookup by repeatedly hitting users/1 (some will exceed 200ms threshold)
for _ in range(15):
    requests.get("http://localhost:8000/users/1", timeout=5)
print("traffic_generated")


## 2. Ожидание индексации

Logstash → ES асинхронно. Подождём и принудительно дёрнем `_refresh`.


In [ ]:
time.sleep(15)  # pipeline latency
ES_URL = "http://localhost:9200"
INDEX = "app-logs-*"

# Force refresh so all buffered docs become searchable right now
requests.post(f"{ES_URL}/{INDEX}/_refresh", timeout=10).raise_for_status()
print("refreshed")


## 3. Сырой запрос логов


In [ ]:
def es_search(body):
    r = requests.post(f"{ES_URL}/{INDEX}/_search", json=body, timeout=15)
    r.raise_for_status()
    return r.json()

body = {"size": 10, "sort": [{"@timestamp": "desc"}], "query": {"match_all": {}}}
res = es_search(body)
print("total hits:", res["hits"]["total"]["value"])
print("\nsample documents:\n")
for i, hit in enumerate(res["hits"]["hits"][:5], 1):
    src = hit["_source"]
    keep = {k: src.get(k) for k in ("@timestamp","level","logger","message","path","method","status_code","latency_ms","username","user_id","request_id","error_type","reason") if k in src}
    print(f"[{i}] {json.dumps(keep, ensure_ascii=False, indent=2)}")

required = {"@timestamp","level","logger","message","path","method","status_code","latency_ms"}
sample = res["hits"]["hits"][0]["_source"] if res["hits"]["hits"] else {}
missing = required - set(sample.keys())
assert not missing, f"missing log fields in sample doc: {missing}"
assert res["hits"]["total"]["value"] > 0, "no documents indexed yet"


## 4. Агрегации Elasticsearch


In [ ]:
agg_body = {
  "size": 0,
  "aggs": {
    "by_level": {"terms": {"field": "level", "size": 10}},
    "by_path": {"terms": {"field": "path", "size": 10}},
    "latency_by_path": {
      "terms": {"field": "path", "size": 10},
      "aggs": {
        "p50": {"percentiles": {"field": "latency_ms", "percents": [50]}},
        "p95": {"percentiles": {"field": "latency_ms", "percents": [95]}},
        "p99": {"percentiles": {"field": "latency_ms", "percents": [99]}},
      },
    },
    "over_time": {
      "date_histogram": {"field": "@timestamp", "fixed_interval": "10s", "min_doc_count": 0},
      "aggs": {
        "by_level": {"terms": {"field": "level", "size": 5}},
      },
    },
  },
}
res = es_search(agg_body)
aggs = res["aggregations"]

def to_df(buckets):
    return pd.DataFrame([{"key": b["key"], "doc_count": b["doc_count"]} for b in buckets])
def percentile_value(bucket, p):
    v = bucket[f"p{p}"]["values"]
    return v.get(f"{p}.0") or list(v.values())[0] if v else 0.0

df_level  = to_df(aggs["by_level"]["buckets"])
df_path   = to_df(aggs["by_path"]["buckets"])
df_lat    = pd.DataFrame([
   {"path": b["key"], "count": b["doc_count"],
    "p50_ms": percentile_value(b, 50), "p95_ms": percentile_value(b, 95),
    "p99_ms": percentile_value(b, 99)}
   for b in aggs["latency_by_path"]["buckets"]
])
df_time   = pd.DataFrame([
   {"t": pd.to_datetime(b["key_as_string"]),
    "level": s["key"], "count": s["doc_count"]}
   for b in aggs["over_time"]["buckets"]
   for s in b["by_level"]["buckets"]
])

print("logs by level:\n", df_level, "\n")
print("logs by endpoint:\n", df_path, "\n")
print("latency by endpoint (ms):\n", df_lat, "\n")
print(f"time buckets: {len(df_time)}")


## 5. Визуализация


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("ELK Demo — log analytics", fontsize=14)

# 1) by level
ax = axes[0, 0]
if len(df_level):
    ax.bar(df_level["key"], df_level["doc_count"], color=["#2ca02c","#ffcc00","#d62728","#1f77b4"][:len(df_level)])
ax.set_title("Logs by level")
ax.set_ylabel("count")
ax.grid(True, axis="y", alpha=0.3)

# 2) by path
ax = axes[0, 1]
if len(df_path):
    ax.barh(df_path["key"], df_path["doc_count"], color="#1f77b4")
ax.set_title("Logs by endpoint")
ax.set_xlabel("count")
ax.grid(True, axis="x", alpha=0.3)
ax.invert_yaxis()

# 3) latency p95 per endpoint
ax = axes[1, 0]
if len(df_lat):
    x = range(len(df_lat))
    ax.bar([i-0.25 for i in x], df_lat["p50_ms"], width=0.25, label="p50", color="#1f77b4")
    ax.bar([i for x_ in x for i in [x_]], df_lat["p95_ms"], width=0.25, label="p95", color="#ff7f0e")
    ax.bar([i+0.25 for i in x], df_lat["p99_ms"], width=0.25, label="p99", color="#d62728")
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_lat["path"], rotation=15, ha="right", fontsize=8)
ax.set_title("Latency by endpoint (ms)")
ax.set_ylabel("ms")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

# 4) over time (per-level lines)
ax = axes[1, 1]
if len(df_time):
    for level_name, sub in df_time.groupby("level"):
        ax.plot(sub["t"], sub["count"], marker="o", linewidth=2, label=level_name)
ax.set_title("Logs over time (10s buckets)")
ax.set_ylabel("count / 10s")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


## 6. Kibana

Дашборд преднастроен через init-контейнер kibana-setup и доступен по адресу ниже.


In [ ]:
kibana_url = "http://localhost:5601/app/dashboards#/view/app-logs-dashboard"
print(f"Kibana UI:   {kibana_url}")
print(f"Discover:    http://localhost:5601/app/discover#/viewedFromIndexPatternId:app-logs-pattern")
print(f"Data view:   app-logs-*")
print(f"ES API:      http://localhost:9200")
print(f"FastAPI:     http://localhost:8000/health")


In [ ]:
summary = """
## Итог

Мы прошли полный цикл журналирования микросервисов:
1. Сгенерировали разнотипную нагрузку на FastAPI (login/users/health).
2. python-logstash-async унёс JSON-логи в Logstash по TCP.
3. Logstash положил их в Elasticsearch (index app-logs-*).
4. Notebook через ES REST API собрал агрегации и построил графики.
5. Kibana UI предоставляет готовый дашборд для ручного изучения.

Стек можно наращивать: Filebeat вместо TCP-инпута, алерты через ElastAlert/Watcher, ретенция через ILM.
"""
print(summary)
